# Build the feature CSV → save it to Google Drive

Clones **[GazeVLM-HWSW-Codesign](https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign)**,
runs the repo's own script on **2 videos**, and saves the result to Drive so it outlives
the Colab session.

```bash
python -m src.dataprep.build_feature_csv --n_videos 2 --cleanup_raw
```

## The CSV alone is not enough

`feat_frame_1` and `feat_frame_2` are **paths** to `.npz` feature files. Saving only the
CSV would leave those paths pointing at `/content/...`, which disappears when the runtime
ends — the file would look fine and be useless.

So this notebook copies **both**, and rewrites the paths inside the CSV to their Drive
locations. The result is self-contained: reload it in any future session and it works.

| Saved to Drive | What it is |
|---|---|
| `feature_dataset.csv` | the table, with Drive paths |
| `features/<sequence>/feat_*.npz` | the frozen DINOv2 features |

## Cost

2 videos ≈ **5 GB** downloaded (deleted afterwards by `--cleanup_raw`), ~13 MB kept.
Expect ~4 min. **No GPU needed.**

## 1 — Clone the repo and install

In [ ]:
!git clone -q https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git /content/GazeVLM
%cd /content/GazeVLM
!git log --oneline -1

!pip -q install -r requirements.txt
!pip -q install projectaria-tools

import os, glob, json, shutil, time
import numpy as np, pandas as pd
print("\nrepo:", os.getcwd())

## 2 — Mount Drive and pick where things go

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/GazeVLM"        # <- change if you like
DRIVE_FEATS = os.path.join(DRIVE_DIR, "features")
DRIVE_CSV   = os.path.join(DRIVE_DIR, "feature_dataset.csv")
os.makedirs(DRIVE_FEATS, exist_ok=True)

print("will save to:")
print("   ", DRIVE_CSV)
print("   ", DRIVE_FEATS + "/<sequence>/feat_*.npz")

## 3 — The download-links JSON

If you already keep it in Drive, use Option B and skip the upload every session.

In [ ]:
# ---- Option A: upload from your laptop ----
from google.colab import files
up = files.upload()                       # pick your aea_download_urls.json
URLS_JSON = "/content/" + list(up.keys())[0]
os.rename(list(up.keys())[0], URLS_JSON)

# ---- Option B: already in Drive ----
# URLS_JSON = os.path.join(DRIVE_DIR, "aea_download_urls.json")

# keep a copy in Drive so future sessions can use Option B
kept = os.path.join(DRIVE_DIR, "aea_download_urls.json")
if os.path.abspath(URLS_JSON) != os.path.abspath(kept):
    shutil.copy2(URLS_JSON, kept)
    print("copied the JSON to Drive for next time:", kept)

print(f"{len(json.load(open(URLS_JSON))['sequences'])} videos available")
print("NOTE: the links expire after ~14 days -- re-download the JSON when builds fail.")

## 4 — Build, on local disk

Built in `/content` rather than straight to Drive: writing hundreds of small `.npz` files
over the Drive mount is far slower. They get copied across in one go afterwards.

Every video is used in full (~190 rows each at 1 FPS), so expect **~380 rows**.

In [ ]:
N_VIDEOS  = 2
SEED      = 0
LOCAL_CSV = "/content/data/feature_dataset.csv"
LOCAL_FEAT = "/content/data/features"

t0 = time.time()
!python -m src.dataprep.build_feature_csv \
    --urls_json  "{URLS_JSON}" \
    --out_csv    "{LOCAL_CSV}" \
    --raw_dir    /content/data/raw \
    --frames_dir /content/data/frames_1fps \
    --feat_dir   "{LOCAL_FEAT}" \
    --n_videos   {N_VIDEOS} \
    --seed       {SEED} \
    --cleanup_raw

print(f"\nbuild took {(time.time()-t0)/60:.1f} min")

## 5 — Copy to Drive and rewrite the paths

This is the step that makes the saved CSV actually reusable.

In [ ]:
# --- features ---------------------------------------------------------------
t0 = time.time()
n_files = 0
for seq in sorted(os.listdir(LOCAL_FEAT)):
    src_dir = os.path.join(LOCAL_FEAT, seq)
    dst_dir = os.path.join(DRIVE_FEATS, seq)
    if not os.path.isdir(src_dir):
        continue
    os.makedirs(dst_dir, exist_ok=True)
    for f in sorted(os.listdir(src_dir)):
        shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
        n_files += 1
    print(f"   {seq}: {len(os.listdir(dst_dir))} files")
print(f"copied {n_files} feature files in {time.time()-t0:.0f}s")

# --- CSV, with paths pointing at Drive --------------------------------------
df = pd.read_csv(LOCAL_CSV)

def to_drive(p):
    """/content/data/features/<seq>/feat_x.npz -> <DRIVE_FEATS>/<seq>/feat_x.npz
    Keeps the <sequence>/<file> tail: the basename alone is NOT unique, since every
    sequence folder contains a feat_00000.npz."""
    parts = str(p).replace("\\", "/").rstrip("/").split("/")
    return os.path.join(DRIVE_FEATS, *parts[-2:])

for col in ("feat_frame_1", "feat_frame_2"):
    df[col] = df[col].apply(to_drive)

df.to_csv(DRIVE_CSV, index=False)
print(f"\nwrote {len(df)} rows -> {DRIVE_CSV}")
print(f"   {os.path.getsize(DRIVE_CSV)/1e3:.0f} KB")
print(f"\nexample rewritten path:\n   {df.loc[0, 'feat_frame_1']}")

## 6 — Verify what landed in Drive

Reads the CSV **back from Drive**, opens the feature files it points at, and recomputes a
similarity from them. If this passes, the saved dataset is self-contained.

In [ ]:
chk = pd.read_csv(DRIVE_CSV)
print(f"{len(chk)} rows x {chk.shape[1]} columns from {chk['sequence'].nunique()} videos")
display(chk.groupby("sequence").size().rename("rows").to_frame())

missing = [p for p in pd.concat([chk.feat_frame_1, chk.feat_frame_2]).unique()
           if not os.path.exists(p)]
print(f"\nreferenced feature files missing: {len(missing)}")

z0 = np.load(chk.loc[0, "feat_frame_1"])
z1 = np.load(chk.loc[0, "feat_frame_2"])
fs = float(np.dot(z0["cls"], z1["cls"]))            # tokens are L2-normalised

g = int(z0["grid"])
def cell(z):
    x, y = z["gaze_xy"]
    gx = min(g-1, int(np.clip(x, 0, 1)*g)); gy = min(g-1, int(np.clip(y, 0, 1)*g))
    return z["patches"][gy*g + gx]
ps = float(np.dot(cell(z0), cell(z1)))

ok = (len(missing) == 0
      and abs(fs - chk.loc[0, "frame_similarity"]) < 1e-3
      and abs(ps - chk.loc[0, "gaze_patch_token_sim"]) < 1e-3)
print(f"\nrecomputed from Drive vs the CSV:")
print(f"   frame_similarity     {fs:.4f}  vs  {chk.loc[0,'frame_similarity']:.4f}")
print(f"   gaze_patch_token_sim {ps:.4f}  vs  {chk.loc[0,'gaze_patch_token_sim']:.4f}")
print(f"\n   [{'PASS' if ok else 'FAIL'}]  the dataset in Drive is self-contained")

sz = sum(os.path.getsize(os.path.join(r, f))
         for r, _, fs_ in os.walk(DRIVE_FEATS) for f in fs_) / 1e6
print(f"\nDrive usage: {sz:.1f} MB of features + {os.path.getsize(DRIVE_CSV)/1e3:.0f} KB CSV")
display(chk.head(5))

---

## Using it in a later session

```python
from google.colab import drive; drive.mount("/content/drive")
CSV = "/content/drive/MyDrive/GazeVLM/feature_dataset.csv"
```

```bash
python -m src.loss1.train --csv $CSV --out_dir runs/loss1
python -m src.loss2.train --csv $CSV --out_dir runs/loss2
```

No rebuild, no re-download, no DINOv2 — the paths already point into Drive.

If you ever move the `features/` folder, pass `--feat_root <new location>` instead of
rewriting the CSV; the loaders re-root on the `<sequence>/<file>` tail.

## Adding more videos later

Change `SEED` (or pass `--seqs` with explicit names), rerun, and **write to a different
`--out_csv`**, then concatenate:

```python
big = pd.concat([pd.read_csv(a), pd.read_csv(b)], ignore_index=True)
big["idx"] = range(len(big))          # idx must stay unique and contiguous
big.to_csv("/content/drive/MyDrive/GazeVLM/feature_dataset.csv", index=False)
```

Feature folders are named per sequence, so they merge without collisions.

## A note on scale

2 videos gives ~380 rows and a 1 train / 1 val split — enough to check the plumbing, not
enough to train anything meaningful. The models carry ~1.5 M parameters. Build up to tens
of videos in Drive before drawing conclusions from any training run.